# 02 — Tiền xử lý & đặc trưng

**Notebook gốc:** bước 2–3: kiểm tra chất lượng, làm sạch benign train (thay ô lỗi bằng 0, giữ nguyên dòng), signed-log, loại feature hằng/hiếm/tương quan, chuẩn hóa. Các hàm nằm trong `src/preprocessing.py`. Notebook này là **kiểm tra độc lập**; notebook 03 sẽ fit lại pipeline khi chạy toàn bộ để bảo đảm tái lập.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Mở thư mục project trên VS Code, hoặc chạy notebook trong notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import config as cfg
print("Project:", PROJECT_ROOT)
print("Dataset:", cfg.DATA_DIR)


In [ ]:
import joblib
from pyspark.sql import functions as F
from src.spark_session import create_spark
from src.data_io import prepare_curated
from src.preprocessing import profile, sanitize_features, fit_feature_spec
from src.utils import day_of

(cfg.OUT_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(cfg.OUT_DIR / 'models').mkdir(parents=True, exist_ok=True)
spark = create_spark()
files = cfg.TRAIN_FILES + [cfg.VAL_FILE] + cfg.TEST_FILES
all_df = prepare_curated(spark, files)
feats = [c for c in all_df.columns if c not in cfg.EXCLUDE_FEATURES]
train_days = [day_of(p) for p in cfg.TRAIN_FILES]
ben_train_raw = all_df.filter(F.col('day').isin(train_days) & (F.col('y') == 0))
quality = profile(ben_train_raw, feats)
display(quality.sort_values('nonfinite', ascending=False).head(20))
quality.to_csv(cfg.OUT_DIR / 'metrics' / 'data_quality_train.csv', index=False)


In [ ]:
ben_train = sanitize_features(ben_train_raw, feats)
print('Benign ban đầu:', ben_train_raw.count(), '| hợp lệ:', ben_train.count())
cols, mean, std, feature_report = fit_feature_spec(ben_train, feats)
display(feature_report)
feature_report.to_csv(cfg.OUT_DIR / 'metrics' / 'feature_report.csv', index=False)
joblib.dump({'cols': cols, 'mean': mean, 'std': std, 'z_clip': cfg.Z_CLIP},
            cfg.OUT_DIR / 'models' / 'feature_spec.joblib')
print(f'Giữ {len(cols)}/{len(feats)} đặc trưng')


**Quan trọng:** Không fit bước chọn/chuẩn hóa đặc trưng lên validation hoặc test. Cả notebook này và `run_pipeline.py` đều áp dụng cách đó.

In [ ]:
spark.stop()